# Before the Transformer: squeeze, then attend

Today's lecture told a story that starts with a problem: **translation**. This notebook lets you touch the two ideas that opened it, with real data and a real metric.

| Section | Year | The idea | The slide it comes from |
|---|---|---|---|
| 1 · One vector | 2014 | squeeze the whole sentence into one vector, then write the translation from it | *squeeze the sentence into one vector* |
| 2 · Attention | 2015 | let the decoder look back at every input word, with weights | *attention, as a patch* |

We stop right before 2017, on purpose. On Wednesday we build the Transformer piece by piece; today you will feel **why** it had to exist.

Runs on a free Colab GPU in about five minutes. `Runtime → Change runtime type → GPU`.

## 🎯 Learning objectives

1. **Train** a 2014-style encoder-decoder translator, reusing the `Architecture` class from lesson 3b.
2. **Measure** the fixed-vector bottleneck: quality as a function of sentence length.
3. **Add** attention and watch the same measurement change.
4. **Read** an alignment map: where the model looked while writing each word.

# 0. The task, the data, the metric

Before 2017, the task that drove the whole field was **machine translation**. So that is our case study: translate English into Portuguese.

- **Data.** [Tatoeba](https://tatoeba.org), a crowd-sourced collection of everyday sentences with translations, packaged by manythings.org. About 170 thousand English–Portuguese pairs, like *"I have a red car." → "Eu tenho um carro vermelho."*
- **Metric.** **BLEU** (0 to 100): how much the words and short phrases of our translation overlap with a human reference. It is the exact metric the Transformer paper reported in 2017, so the number you compute today is the number you will see in the paper.

In [2]:
%%capture
!pip install -q sacrebleu

In [3]:
import os, re, time, random, zipfile, urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import sacrebleu
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch.nn")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device)

PyTorch 2.11.0+cu128 | device: cuda


In [4]:
# Same plotting conventions as lesson 04: quiet axes, one question per chart
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.8,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11, "axes.titlelocation": "left",
})
C_MAIN, C_ACCENT, C_GRAY = "#2E5E9E", "#D95F02", "#9AA0A6"

## 0.1 · Download and first look

In [6]:
DATA_URL = "https://www.manythings.org/anki/por-eng.zip"      # English -> Portuguese
# For the exercises: swap the pair, delete por.txt / *.zip, and rerun from here.
# DATA_URL = "https://www.manythings.org/anki/fra-eng.zip"    # English -> French
# DATA_URL = "https://www.manythings.org/anki/spa-eng.zip"    # English -> Spanish
# DATA_URL = "https://www.manythings.org/anki/deu-eng.zip"    # English -> German
# DATA_URL = "https://www.manythings.org/anki/ita-eng.zip"    # English -> Italian
# full list of pairs: https://www.manythings.org/anki/

ZIP_NAME = DATA_URL.split("/")[-1]                 # e.g. por-eng.zip
TXT_NAME = ZIP_NAME.split("-")[0] + ".txt"         # e.g. por.txt

if not os.path.exists(TXT_NAME):
    # Use !wget for more robust downloading in Colab
    !wget -q $DATA_URL
    with zipfile.ZipFile(ZIP_NAME) as z:
        z.extractall(".")

# columns: English sentence, translation (the third column is attribution, ignored)
pairs = pd.read_csv(TXT_NAME, sep="\t", header=None, usecols=[0, 1], names=["en", "pt"], quoting=3)
print(f"{len(pairs):,} sentence pairs")
pairs.sample(6, random_state=7)

197,147 sentence pairs


,en,pt
97021,Money doesn't grow on trees.,Dinheiro não dá em árvore.
130644,I don't like to swim in the pool.,Eu não gosto de nadar na piscina.
172972,"You've never told Tom the truth, have you?","Vocês nunca contaram a verdade para o Tom, não é?"
41261,Go to the barbershop.,Vá à barbearia.
5448,We'll get it.,Nós vamos buscá-lo.
29565,I'm not your slave.,Não sou seu escravo!


Short, everyday sentences. Good news: models trained on this fit in a coffee break.

Two decisions before modeling, both to keep today's models simple.

- **Word-level tokens.** Lowercase, split punctuation off. (Real LLMs use *subword* tokens; that is the first thing we build on Wednesday.)
- **Length cap of 12 tokens per side.** Keeps training fast, and still leaves plenty of longer sentences for the experiment in 1.4.

In [ ]:
def tokenize(text):
    text = text.lower().strip()
    text = re.sub(r"([.!?,;:¿¡\"()])", r" \1 ", text)   # "car." -> "car ."
    return text.split()

MAX_LEN = 12
pairs["en_tok"] = pairs["en"].map(tokenize)
pairs["pt_tok"] = pairs["pt"].map(tokenize)
mask = (pairs["en_tok"].str.len() <= MAX_LEN) & (pairs["pt_tok"].str.len() <= MAX_LEN)
pairs = pairs[mask].drop_duplicates(subset=["en"]).reset_index(drop=True)
print(f"{len(pairs):,} pairs after the length cap and de-duplication")

## 0.2 · Split, vocabulary, batches

Same discipline as the Airbnb notebook: the test set is set aside **now** and every model will be scored on the same 1,000 sentences. Vocabularies are built **only from the training split** — the same no-leakage rule as the z-score in lesson 04.

Three special tokens do the bookkeeping:

- `<pad>` fills the empty space when we batch sentences of different lengths (the loss will ignore it);
- `<sos>` / `<eos>` wrap every Portuguese sentence, so the decoder knows where to **start** and when to **stop**.

In [ ]:
PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"
PAD_ID, SOS_ID, EOS_ID = 0, 1, 2

class Vocab:
    def __init__(self, sentences, min_freq=2):
        counter = Counter(tok for sent in sentences for tok in sent)
        self.itos = [PAD, SOS, EOS, UNK] + sorted(w for w, c in counter.items() if c >= min_freq)
        self.stoi = {w: i for i, w in enumerate(self.itos)}
    def encode(self, tokens):                       # words -> ids
        return [self.stoi.get(t, self.stoi[UNK]) for t in tokens]
    def decode(self, ids):                          # ids -> words, stopping at <eos>
        out = []
        for i in ids:
            w = self.itos[i]
            if w == EOS: break
            if w not in (PAD, SOS): out.append(w)
        return out
    def __len__(self): return len(self.itos)

rng = np.random.RandomState(13)
idx = rng.permutation(len(pairs))
test_df  = pairs.iloc[idx[:1000]].reset_index(drop=True)
val_df   = pairs.iloc[idx[1000:2000]].reset_index(drop=True)
train_df = pairs.iloc[idx[2000:]].reset_index(drop=True)

src_vocab = Vocab(train_df["en_tok"])
tgt_vocab = Vocab(train_df["pt_tok"])
print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}")
print(f"English vocab {len(src_vocab):,} | Portuguese vocab {len(tgt_vocab):,}")

In [ ]:
class TranslationDataset(Dataset):
    # x = the English sentence | y = the Portuguese one, wrapped in <sos> ... <eos>
    def __init__(self, df):
        self.src = [torch.tensor(src_vocab.encode(t)) for t in df["en_tok"]]
        self.tgt = [torch.tensor([SOS_ID] + tgt_vocab.encode(t) + [EOS_ID]) for t in df["pt_tok"]]
    def __len__(self): return len(self.src)
    def __getitem__(self, i): return self.src[i], self.tgt[i]

def collate(batch):
    # pad every sentence in the batch to the longest one
    src, tgt = zip(*batch)
    src = nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=PAD_ID)
    tgt = nn.utils.rnn.pad_sequence(tgt, batch_first=True, padding_value=PAD_ID)
    return src, tgt

train_loader = DataLoader(TranslationDataset(train_df), batch_size=128, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(TranslationDataset(val_df),   batch_size=128, shuffle=False, collate_fn=collate)

xb, yb = next(iter(train_loader))
print("one batch | x:", tuple(xb.shape), "| y:", tuple(yb.shape))
print("x[0]:", src_vocab.decode(xb[0].tolist()))
print("y[0]:", [tgt_vocab.itos[i] for i in yb[0].tolist() if i != PAD_ID])

## 0.3 · `Architecture`, with two methods overridden

The training class from lesson 3b assumed `model(x)`. A translator trains with a trick called **teacher forcing**: the model reads the English sentence `x` *and* the correct Portuguese answer shifted by one position, and at every position it must predict the **next** word.

```
decoder input :  <sos>  eu    tenho  um    carro  vermelho
target        :  eu    tenho  um    carro  vermelho  <eos>
```

That changes exactly two methods — the train step and the validation step. The loaders, the epoch loop and the loss plot are inherited untouched. This is the payoff of writing `Architecture` once in lesson 3b.

In [ ]:
class Architecture(object):
    # lesson 3b class, unchanged (trimmed to what this notebook uses)
    def __init__(self, model, loss_fn, optimizer):
        self.model, self.loss_fn, self.optimizer = model, loss_fn, optimizer
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        self.train_loader = self.val_loader = None
        self.losses, self.val_losses, self.total_epochs = [], [], 0
        self.train_step_fn = self._make_train_step_fn()
        self.val_step_fn = self._make_val_step_fn()

    def set_loaders(self, train_loader, val_loader=None):
        self.train_loader, self.val_loader = train_loader, val_loader

    def _make_train_step_fn(self):
        def perform_train_step_fn(x, y):
            self.model.train()
            yhat = self.model(x)
            loss = self.loss_fn(yhat, y)
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()
            return loss.item()
        return perform_train_step_fn

    def _make_val_step_fn(self):
        def perform_val_step_fn(x, y):
            self.model.eval()
            yhat = self.model(x)
            return self.loss_fn(yhat, y).item()
        return perform_val_step_fn

    def _mini_batch(self, validation=False):
        data_loader = self.val_loader if validation else self.train_loader
        step_fn = self.val_step_fn if validation else self.train_step_fn
        if data_loader is None: return None
        mini_batch_losses = []
        for x_batch, y_batch in data_loader:
            x_batch, y_batch = x_batch.to(self.device), y_batch.to(self.device)
            mini_batch_losses.append(step_fn(x_batch, y_batch))
        return np.mean(mini_batch_losses)

    def set_seed(self, seed=42):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    def train(self, n_epochs, seed=42):
        self.set_seed(seed)
        for epoch in range(n_epochs):
            self.total_epochs += 1
            self.losses.append(self._mini_batch(validation=False))
            with torch.no_grad():
                self.val_losses.append(self._mini_batch(validation=True))

    def plot_losses(self):
        fig = plt.figure(figsize=(10, 4))
        plt.plot(self.losses, label='Training Loss', c=C_MAIN)
        plt.plot(self.val_losses, label='Validation Loss', c=C_ACCENT)
        plt.yscale('log'); plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend(); plt.tight_layout()
        return fig


class Seq2SeqArchitecture(Architecture):
    # The ONLY change: the model sees x AND the target shifted by one (teacher forcing)
    def _make_train_step_fn(self):
        def perform_train_step_fn(x, y):
            self.model.train()
            logits = self.model(x, y[:, :-1])            # feed  <sos> w1 ... w_{n-1}
            loss = self.loss_fn(logits.reshape(-1, logits.size(-1)),
                                y[:, 1:].reshape(-1))    # predict  w1 ... <eos>
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)   # standard for RNNs
            self.optimizer.step()
            self.optimizer.zero_grad()
            return loss.item()
        return perform_train_step_fn

    def _make_val_step_fn(self):
        def perform_val_step_fn(x, y):
            self.model.eval()
            logits = self.model(x, y[:, :-1])
            return self.loss_fn(logits.reshape(-1, logits.size(-1)), y[:, 1:].reshape(-1)).item()
        return perform_val_step_fn

Two evaluation helpers, used by **every** model today, so comparisons are fair.

- `translate` decodes **greedily**: start from `<sos>`, always append the most likely next word, stop at `<eos>`.
- `bleu_on` scores 1,000 test sentences at once.

In [ ]:
@torch.no_grad()
def translate_batch(model, src, max_len=MAX_LEN + 2):
    model.eval()
    src = src.to(device)
    ys = torch.full((src.size(0), 1), SOS_ID, dtype=torch.long, device=device)
    for _ in range(max_len):
        logits = model(src, ys)                              # (batch, steps so far, vocab)
        next_id = logits[:, -1].argmax(-1, keepdim=True)     # most likely next word
        ys = torch.cat([ys, next_id], dim=1)
        if (ys == EOS_ID).any(dim=1).all(): break            # every sentence has ended
    return [tgt_vocab.decode(row.tolist()[1:]) for row in ys]

def translate(model, sentence):
    src = torch.tensor([src_vocab.encode(tokenize(sentence))])
    return " ".join(translate_batch(model, src)[0])

def bleu_on(model, df, batch_size=256):
    hyps = []
    for i in range(0, len(df), batch_size):
        chunk = df.iloc[i:i + batch_size]
        src = nn.utils.rnn.pad_sequence([torch.tensor(src_vocab.encode(t)) for t in chunk["en_tok"]],
                                        batch_first=True, padding_value=PAD_ID)
        hyps += [" ".join(h) for h in translate_batch(model, src)]
    refs = [" ".join(t) for t in df["pt_tok"]]
    return sacrebleu.corpus_bleu(hyps, [refs], tokenize="none", force=True).score

# 1. 2014 · Squeeze the sentence into one vector

Sutskever, Vinyals and Le (Google, 2014) proposed **seq2seq**: two LSTMs.

```
the  ->  cat  ->  sat            [one vector]            le -> chat -> ...
└──────  encoder  ──────┘  ->  (h, c)  ->  └──────  decoder  ──────┘
         reads                  the whole            writes, one word
                                sentence             at a time
```

- The **encoder** reads English word by word and ends with one hidden state.
- The **decoder** starts from that state and writes Portuguese, one word at a time.
- The *only* thing crossing the bridge is `(h, c)` — a fixed-size vector, whether the sentence has 3 words or 12.

That last point is the whole story of this section. Look for it in the code: `encode` returns just `(h, c)`, and `forward` gives the decoder nothing else.

In [ ]:
class Seq2Seq(nn.Module):
    # 2014: encoder LSTM -> one vector -> decoder LSTM
    def __init__(self, src_vocab_size, tgt_vocab_size, emb=256, hid=512):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab_size, emb, padding_idx=PAD_ID)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, emb, padding_idx=PAD_ID)
        self.encoder = nn.LSTM(emb, hid, batch_first=True)
        self.decoder = nn.LSTM(emb, hid, batch_first=True)
        self.out = nn.Linear(hid, tgt_vocab_size)

    def encode(self, src):
        lengths = (src != PAD_ID).sum(1).cpu()               # real length of each sentence
        packed = nn.utils.rnn.pack_padded_sequence(self.src_emb(src), lengths,
                                                   batch_first=True, enforce_sorted=False)
        _, (h, c) = self.encoder(packed)
        return h, c            # <- the whole sentence, squeezed into one fixed-size state

    def forward(self, src, tgt_in):
        h, c = self.encode(src)
        dec_out, _ = self.decoder(self.tgt_emb(tgt_in), (h, c))
        return self.out(dec_out)                             # (batch, steps, vocab) logits

### 🔮 Predict

A professional system scores BLEU above 50 on sentences this easy. A 2014-style model trained for two minutes — what do you expect? Write a number down before running.

In [ ]:
%%time
torch.manual_seed(42)
seq2seq = Seq2Seq(len(src_vocab), len(tgt_vocab))
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)     # padding does not count as error

arch_s2s = Seq2SeqArchitecture(seq2seq, loss_fn, optim.Adam(seq2seq.parameters(), lr=1e-3))
arch_s2s.set_loaders(train_loader, val_loader)
arch_s2s.train(n_epochs=6)

bleu_s2s = bleu_on(seq2seq, test_df)
print(f"BLEU on 1,000 test sentences: {bleu_s2s:.1f}")

In [ ]:
fig = arch_s2s.plot_losses()

In [ ]:
for s in ["a red car .",
          "i have a red car .",
          "my sister bought a red car yesterday ."]:
    print(f"{s:45s} -> {translate(seq2seq, s)}")

### 🔍 Reading the output

The short sentence comes out fine. As the sentence grows, words get dropped or the ending drifts. That is not bad luck — it is the single vector filling up. Six words or twelve, the decoder receives the same 512 numbers.

An anecdote from the slide: the whole sentence, squeezed. **5 words or 50 words, same 512 numbers.** Let's turn that suspicion into a measurement.

## 1.4 · The bottleneck, measured

**Question.** Does quality fall as the input gets longer?

We split the same test set into four buckets by English sentence length and compute BLEU per bucket.

In [ ]:
def bleu_by_length(model, df):
    bins = [(1, 4), (5, 6), (7, 8), (9, 12)]
    rows = []
    for lo, hi in bins:
        sub = df[df["en_tok"].str.len().between(lo, hi)]
        rows.append((f"{lo}-{hi}", bleu_on(model, sub), len(sub)))
    return pd.DataFrame(rows, columns=["source length", "BLEU", "n sentences"])

by_len_s2s = bleu_by_length(seq2seq, test_df)
by_len_s2s

Hold that table. We will plot it right next to the 2015 model, so the comparison lands in one picture.

# 2. 2015 · Attention, as a patch

Bahdanau, Cho and Bengio (2015) kept the two LSTMs and changed **one thing**. Instead of receiving a single squeezed vector, at *every* step the decoder:

1. looks at **all** the encoder states (one per English word);
2. scores each one against what it is about to write;
3. turns the scores into weights with a softmax (they sum to 1);
4. feeds the weighted mix — the *context* — into the LSTM cell, together with the previous word.

So when the decoder is about to write *vermelho*, it can put weight 0.8 on *red* and almost nothing on the rest. **No single bottleneck vector.** The paper called the weights an *alignment*; the field soon started calling the whole mechanism *attention*.

In the code, spot the two changes against `Seq2Seq`:
- `encode` now returns **all** states, not just the last;
- `forward` still walks one word at a time, but calls `self.attn(...)` at every step.

In [ ]:
class Attention(nn.Module):
    # Bahdanau (additive) attention: score(h_dec, h_enc) = v . tanh(W [h_dec ; h_enc])
    def __init__(self, hid):
        super().__init__()
        self.W = nn.Linear(2 * hid, hid)
        self.v = nn.Linear(hid, 1, bias=False)

    def forward(self, dec_h, enc_out, src_mask):
        # dec_h: (B, H) what we're about to write | enc_out: (B, S, H) every input word
        dec_rep = dec_h.unsqueeze(1).expand(-1, enc_out.size(1), -1)
        scores = self.v(torch.tanh(self.W(torch.cat([dec_rep, enc_out], -1)))).squeeze(-1)  # (B, S)
        scores = scores.masked_fill(~src_mask, -1e9)          # padding never gets attention
        weights = F.softmax(scores, dim=-1)                   # (B, S), sums to 1
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)   # weighted mix, (B, H)
        return context, weights


class Seq2SeqAttn(nn.Module):
    # 2015: same two LSTMs, plus attention over every encoder state
    def __init__(self, src_vocab_size, tgt_vocab_size, emb=256, hid=512):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab_size, emb, padding_idx=PAD_ID)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, emb, padding_idx=PAD_ID)
        self.encoder = nn.LSTM(emb, hid, batch_first=True)
        self.decoder = nn.LSTMCell(emb + hid, hid)   # input: previous word + context
        self.attn = Attention(hid)
        self.out = nn.Linear(hid, tgt_vocab_size)
        self.last_weights = None                     # saved for the alignment plot

    def encode(self, src):
        lengths = (src != PAD_ID).sum(1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(self.src_emb(src), lengths,
                                                   batch_first=True, enforce_sorted=False)
        out, (h, c) = self.encoder(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        return out, h[0], c[0]                       # ALL the states, not just the last

    def forward(self, src, tgt_in):
        enc_out, h, c = self.encode(src)
        src_mask = src != PAD_ID
        emb = self.tgt_emb(tgt_in)
        logits, weights = [], []
        for t in range(tgt_in.size(1)):              # still one word at a time!
            context, w = self.attn(h, enc_out, src_mask)
            h, c = self.decoder(torch.cat([emb[:, t], context], -1), (h, c))
            logits.append(self.out(h)); weights.append(w)
        self.last_weights = torch.stack(weights, dim=1)      # (B, T, S)
        return torch.stack(logits, dim=1)

In [ ]:
%%time
torch.manual_seed(42)
s2s_attn = Seq2SeqAttn(len(src_vocab), len(tgt_vocab))

arch_attn = Seq2SeqArchitecture(s2s_attn, loss_fn, optim.Adam(s2s_attn.parameters(), lr=1e-3))
arch_attn.set_loaders(train_loader, val_loader)
arch_attn.train(n_epochs=6)

bleu_attn = bleu_on(s2s_attn, test_df)
print(f"BLEU on 1,000 test sentences: {bleu_attn:.1f}   (2014 model: {bleu_s2s:.1f})")

## 2.2 · The same measurement, side by side

Same test set, same buckets, one line per model. This chart is the argument of the whole section.

In [ ]:
by_len_attn = bleu_by_length(s2s_attn, test_df)

fig, ax = plt.subplots(figsize=(8, 4.2))
for name, df_, color in [("2014 · one vector", by_len_s2s, C_GRAY), ("2015 · + attention", by_len_attn, C_MAIN)]:
    ax.plot(df_["source length"], df_["BLEU"], marker="o", color=color, linewidth=2, label=name)
    ax.annotate(f"{df_['BLEU'].iloc[-1]:.0f}", xy=(3, df_["BLEU"].iloc[-1]), xytext=(8, 0),
                textcoords="offset points", color=color, fontweight="bold", va="center")
ax.set_title("BLEU by sentence length: one vector versus attention")
ax.set_xlabel("English sentence length (tokens)")
ax.set_ylabel("BLEU on test")
ax.legend(frameon=False)
plt.tight_layout()

### 🔍 What the curves say

Both models were trained for the same six epochs on the same data. The 2014 model loses more as sentences grow; the attention model holds. The only difference is **where the information lives**: one squeezed vector, or every encoder state, mixed on demand.

## 2.3 · The alignment map: the slide, generated by you

The weights the attention computed are not hidden — we saved them in `last_weights`. Rows are the Portuguese words the decoder wrote; columns are the English words it looked at; each row sums to 1.

Watch the adjective. English says *red car*; Portuguese says *carro vermelho*, in the opposite order.

In [ ]:
def plot_alignment(model, sentence):
    src_tokens = tokenize(sentence)
    src = torch.tensor([src_vocab.encode(src_tokens)]).to(device)
    out_tokens = translate_batch(model, src)[0]
    # run the decoder once more on its own output, just to read the weights
    tgt_in = torch.tensor([[SOS_ID] + tgt_vocab.encode(out_tokens)]).to(device)
    with torch.no_grad():
        model.eval(); model(src, tgt_in)
    w = model.last_weights[0, :len(out_tokens), :len(src_tokens)].cpu().numpy()

    fig, ax = plt.subplots(figsize=(1.1 * len(src_tokens) + 2, 1.0 * len(out_tokens) + 1.5))
    ax.imshow(w, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(src_tokens))); ax.set_xticklabels(src_tokens)
    ax.set_yticks(range(len(out_tokens))); ax.set_yticklabels(out_tokens)
    ax.set_title("Where the decoder looked while writing each word")
    ax.grid(False)
    for i in range(w.shape[0]):
        for j in range(w.shape[1]):
            if w[i, j] > 0.15:
                ax.text(j, i, f"{w[i, j]:.2f}", ha="center", va="center",
                        color="white" if w[i, j] > 0.5 else "black", fontsize=9)
    plt.tight_layout()
    print("translation:", " ".join(out_tokens))

plot_alignment(s2s_attn, "a red car .")

In [ ]:
plot_alignment(s2s_attn, "i think my sister bought a red car yesterday .")

✅ **Check yourself**

1. The attention model has more parameters than the plain one. Could the gain in 2.2 be size alone? Sketch the experiment that would settle it.
2. In the first alignment map, which English column gets the weight when the model writes *vermelho*? What does that say about word order?
3. `Seq2SeqAttn.forward` still contains `for t in range(...)`. Attention fixed the bottleneck — did it fix the **speed**?

# 3. Where this leaves us

Two measurements to take home:

- **One vector is a bottleneck.** Quality falls with length, because 5 words or 50 words get the same 512 numbers.
- **Attention removes it.** Let the decoder look back at every input word, with learned weights, and the fall flattens. You can even *see* the weights.

But look again at question 3 above. The decoder still writes **one word at a time, inside a Python loop**. Attention patched the bottleneck; it did not touch the slowness. Word *n* still waits for word *n − 1*, and the GPU — a machine built for parallelism — sits mostly idle.

In 2017, eight people at Google asked the question this notebook has been building toward:

> *if attention is doing the real work — what if we throw away the recurrence and keep only the attention?*

The answer is called the **Transformer**, and on Wednesday we build it piece by piece: tokenization, embeddings, and the attention you just watched, reorganized so that everything happens at once.

## 🏋️ Exercises

No solutions here on purpose. Each one reuses the notebook as-is.

**1 · Swap the language.**
Point `DATA_URL` to one of the commented pairs at the top (French, Spanish, German, Italian), delete the old `.txt` and `.zip`, and rerun everything. Before running, write a prediction: will French score higher or lower than Portuguese, and why? Record your three numbers, seq2seq BLEU, attention BLEU, and the length chart.
*Warning that is actually a lesson: languages written without spaces, like Japanese or Chinese, break our word-level tokenizer. Wednesday's subword tokenization is the fix.*

**2 · Compare with a colleague who picked a different pair.**
Raw BLEU is not comparable across languages, but the *shape* of the length chart is. Does the 2014 curve fall for every language? Does attention flatten it in every case? Write one sentence stating what that says about where the bottleneck lives, the architecture or the language.

**3 · Make the bottleneck worse on purpose.**
Set `MAX_LEN = 20`, rebuild the data cells, and retrain only the 2014 `Seq2Seq`. Before training, predict whether the new 13–20 bucket will lose more or less BLEU than the 9–12 one did. Then plot `bleu_by_length` with an extra bin and check.

**4 · Read a foreign alignment.**
In your chosen pair, find a sentence where word order differs from English, the adjective in French, the verb at the end in German, and call `plot_alignment` on it. Does the attention cross exactly where the grammar crosses?

## 📚 To go deeper

- Sutskever, Vinyals, Le. *Sequence to Sequence Learning with Neural Networks* (NeurIPS 2014) — Section 1 is very readable.
- Bahdanau, Cho, Bengio. *Neural Machine Translation by Jointly Learning to Align and Translate* (ICLR 2015) — Figure 3 is the alignment map you just generated.
- Vaswani et al. *Attention Is All You Need* (2017) — just the title, for now. Wednesday we read the rest.